# 🛒 Retail Sales Deep Learning Pipeline
### Multi-Agent System: Forecasting + Anomaly Detection

This notebook runs the full pipeline end-to-end:
1. **Setup** – Install dependencies & upload data
2. **Preprocessing Agent** – Merge, clean, and feature-engineer the dataset
3. **Training Pipeline** – Train LSTM Forecasting + LSTM Autoencoder Anomaly models
4. **Inference Pipeline** – Run predictions and anomaly detection
5. **Recommendation Agent** – Generate rule-based business recommendations
6. **Human-in-the-Loop Agent** – Validate recommendations
7. **Report Agent** – Generate the final business report
8. **Visualization** – View all plots and results

> **GPU recommended**: Go to *Runtime → Change runtime type → T4 GPU*

## ⚙️ Section 1 – Install Dependencies

In [ ]:
%%capture
!pip install pandas numpy scikit-learn torch matplotlib crewai crewai-tools litellm python-dotenv langchain-community

In [ ]:
import os, json, pickle, random, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datetime import datetime
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, Image as IPImage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded | Device: {device}')

## 📂 Section 2 – Upload & Prepare Data

Upload your dataset. You have **two options**:

**Option A** – Use the Kaggle "Store Sales" dataset (recommended for the full demo)

**Option B** – Upload your own CSV file

In [ ]:
# ─── Option A: Download the Store Sales dataset from Kaggle ───────────────────
# Uncomment this block if you have a Kaggle API key (kaggle.json)

# from google.colab import files
# print('Upload your kaggle.json API key:')
# files.upload()
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c store-sales-time-series-forecasting -p data/store_sales
# !unzip -q data/store_sales/store-sales-time-series-forecasting.zip -d data/store_sales/
# DATA_PATH = 'data/store_sales'

In [ ]:
# ─── Option B: Upload your own CSV (or the provided sample files) ─────────────
# You can upload one or more CSV files. The pipeline auto-detects the main file.

from google.colab import files

print('📤 Upload one or more CSV files (train.csv, transactions.csv, stores.csv, etc.):')
uploaded = files.upload()

os.makedirs('data/uploaded', exist_ok=True)
for fname, data in uploaded.items():
    fpath = f'data/uploaded/{fname}'
    with open(fpath, 'wb') as f:
        f.write(data)
    print(f'  ✅ Saved: {fpath}')

DATA_PATH = 'data/uploaded'
print(f'\n📁 Data path set to: {DATA_PATH}')

In [ ]:
# ─── Quick peek at uploaded files ────────────────────────────────────────────
for fname in os.listdir(DATA_PATH):
    if fname.endswith('.csv'):
        df_peek = pd.read_csv(f'{DATA_PATH}/{fname}', nrows=3)
        print(f'\n📄 {fname} | Shape: {pd.read_csv(f"{DATA_PATH}/{fname}").shape}')
        display(df_peek)

## 🤖 Section 3 – Agent Definitions

All agent classes are defined below (self-contained, no import issues in Colab).

In [ ]:
# ════════════════════════════════════════════════════════════════
# PREPROCESSING AGENT
# ════════════════════════════════════════════════════════════════

class RetailPreprocessingAgent:
    def __init__(self, input_path, freq='D'):
        self.input_path = input_path
        self.freq = freq
        self.logs = []
        self.dataframes = {}
        self.df = None
        self.main_file = None
        self.date_col = None
        self.target_col = None
        self.metadata = {}

    def log_action(self, action, status, details=None):
        self.logs.append({'timestamp': datetime.now().isoformat(),
                          'agent': 'RetailPreprocessingAgent',
                          'action': action, 'status': status, 'details': details})

    def load_data(self):
        if os.path.isfile(self.input_path):
            name = os.path.basename(self.input_path)
            self.dataframes[name] = pd.read_csv(self.input_path)
        elif os.path.isdir(self.input_path):
            for file in os.listdir(self.input_path):
                if file.endswith('.csv'):
                    path = os.path.join(self.input_path, file)
                    self.dataframes[file] = pd.read_csv(path)
        else:
            raise ValueError('Input path must be a CSV file or folder containing CSV files.')
        if not self.dataframes:
            raise ValueError('No CSV files found.')
        self.log_action('load_data', 'success', list(self.dataframes.keys()))
        return self.dataframes

    def detect_date_column(self, df):
        for col in df.columns:
            if any(k in col.lower() for k in ['date', 'time', 'timestamp', 'day']):
                converted = pd.to_datetime(df[col], errors='coerce')
                if converted.notna().mean() > 0.7:
                    return col
        return None

    def detect_target_column(self, df):
        keywords = ['sales', 'revenue', 'amount', 'quantity', 'qty', 'demand']
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        scores = []
        for col in numeric_cols:
            score = 0
            if any(k in col.lower() for k in keywords):
                score += 10
            if 'id' in col.lower() or col.lower() in ['store_nbr', 'product_id', 'item_id']:
                score -= 10
            if df[col].nunique() > 10:
                score += 2
            scores.append((col, score))
        if not scores:
            return None
        return sorted(scores, key=lambda x: x[1], reverse=True)[0][0]

    def choose_main_dataframe(self):
        best_score = -1
        for name, df in self.dataframes.items():
            date_col = self.detect_date_column(df)
            target_col = self.detect_target_column(df)
            score = (5 if date_col else 0) + (5 if target_col else 0) + len(df) / 100000
            if score > best_score:
                best_score = score
                self.main_file = name
        if self.main_file is None:
            raise ValueError('Could not identify main dataset.')
        self.df = self.dataframes[self.main_file].copy()
        self.date_col = self.detect_date_column(self.df)
        self.target_col = self.detect_target_column(self.df)
        if self.date_col is None:
            raise ValueError('No valid date column detected.')
        if self.target_col is None:
            raise ValueError('No valid target column detected.')
        self.log_action('choose_main_dataframe', 'success',
                        {'main_file': self.main_file, 'date_col': self.date_col, 'target_col': self.target_col})

    def merge_secondary_files(self):
        merged = self.df.copy()
        allowed_keys = ['date', 'store_nbr', 'store_id', 'product_id', 'item_id', 'family', 'city', 'state', 'type', 'category']
        for name, other_df in self.dataframes.items():
            if name == self.main_file:
                continue
            common_cols = list(set(merged.columns).intersection(set(other_df.columns)))
            useful_keys = [col for col in common_cols if col.lower() in allowed_keys]
            if not useful_keys:
                self.log_action('merge_secondary_files', 'skipped', f'No useful keys for {name}')
                continue
            try:
                other_df = other_df.copy()
                for col in useful_keys:
                    if 'date' in col.lower():
                        merged[col] = pd.to_datetime(merged[col], errors='coerce')
                        other_df[col] = pd.to_datetime(other_df[col], errors='coerce')
                other_df = other_df.drop_duplicates(subset=useful_keys)
                merged = merged.merge(other_df, on=useful_keys, how='left',
                                      suffixes=('', f'_{name.replace(".csv", "")}'))
                self.log_action('merge_secondary_files', 'success', {'file': name, 'keys': useful_keys})
            except Exception as e:
                self.log_action('merge_secondary_files', 'failed', f'{name}: {e}')
        self.df = merged
        return self.df

    def clean_basic_data(self):
        df = self.df.copy()
        df[self.date_col] = pd.to_datetime(df[self.date_col], errors='coerce')
        df[self.target_col] = pd.to_numeric(df[self.target_col], errors='coerce')
        before = len(df)
        df = df.dropna(subset=[self.date_col, self.target_col])
        df = df[df[self.target_col] >= 0]
        df = df.drop_duplicates()
        df = df.sort_values(self.date_col).reset_index(drop=True)
        self.df = df
        self.log_action('clean_basic_data', 'success', f'Removed {before - len(df)} rows')
        return self.df

    def handle_missing_values(self):
        df = self.df.copy()
        for col in df.select_dtypes(include=[np.number]).columns:
            median_value = df[col].median()
            df[col] = df[col].fillna(0 if pd.isna(median_value) else median_value)
        for col in df.select_dtypes(include=['object', 'bool']).columns:
            df[col] = df[col].fillna('Unknown').astype(str)
        self.df = df
        self.log_action('handle_missing_values', 'success', 'Missing values handled')
        return self.df

    def create_time_features(self):
        df = self.df.copy()
        df['year'] = df[self.date_col].dt.year
        df['month'] = df[self.date_col].dt.month
        df['day'] = df[self.date_col].dt.day
        df['day_of_week'] = df[self.date_col].dt.dayofweek
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        self.df = df
        self.log_action('create_time_features', 'success', 'Time features created')
        return self.df

    def build_model_ready_dataset(self):
        df = self.df.copy()
        df[self.date_col] = pd.to_datetime(df[self.date_col], errors='coerce')
        df[self.target_col] = pd.to_numeric(df[self.target_col], errors='coerce')
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        agg_dict = {col: ('sum' if col == self.target_col else 'mean') for col in numeric_cols}
        model_ready_df = (df.groupby(self.date_col).agg(agg_dict)
                          .reset_index().sort_values(self.date_col))
        model_ready_df = model_ready_df.rename(columns={self.date_col: 'date', self.target_col: 'sales'})
        model_ready_df = model_ready_df.fillna(0)
        self.metadata = {'date_col': 'date', 'target_col': 'sales',
                         'original_date_col': self.date_col, 'original_target_col': self.target_col,
                         'frequency': self.freq, 'main_file': self.main_file,
                         'model_ready_type': 'daily_aggregated_time_series'}
        self.log_action('build_model_ready_dataset', 'success', f'Shape: {model_ready_df.shape}')
        return model_ready_df

    def save_outputs(self, clean_output_path='outputs/clean_merged_data.csv',
                     model_output_path='outputs/model_ready_data.csv',
                     metadata_path='outputs/preprocessing_metadata.json',
                     log_path='outputs/preprocessing_logs.json'):
        os.makedirs('outputs', exist_ok=True)
        clean_df = self.df.copy()
        model_ready_df = self.build_model_ready_dataset()
        clean_df.to_csv(clean_output_path, index=False)
        model_ready_df.to_csv(model_output_path, index=False)
        with open(metadata_path, 'w') as f:
            json.dump(self.metadata, f, indent=4, default=str)
        with open(log_path, 'w') as f:
            json.dump(self.logs, f, indent=4, default=str)
        return clean_df, model_ready_df

    def run(self, clean_output_path='outputs/clean_merged_data.csv',
            model_output_path='outputs/model_ready_data.csv'):
        try:
            self.load_data()
            self.choose_main_dataframe()
            self.merge_secondary_files()
            self.clean_basic_data()
            self.handle_missing_values()
            self.create_time_features()
            clean_df, model_ready_df = self.save_outputs(clean_output_path, model_output_path)
            self.log_action('run_pipeline', 'success', 'Preprocessing completed')
            return clean_df, model_ready_df
        except Exception as e:
            self.log_action('run_pipeline', 'failed', str(e))
            print(f'[ERROR] Preprocessing failed: {e}')
            return None, None

print('✅ RetailPreprocessingAgent defined')

In [ ]:
# ════════════════════════════════════════════════════════════════
# LSTM FORECASTING MODEL + AGENT
# ════════════════════════════════════════════════════════════════

class LSTMForecastingModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, output_size=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(),
                                nn.Dropout(dropout), nn.Linear(32, output_size))

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])


class ForecastingTrendAgent:
    def __init__(self, input_path, window_size=30, future_steps=30,
                 stock_safety_margin=0.15, batch_size=50, scaler_type='robust',
                 threshold_method='hybrid', output_dir='outputs', show_plots=False):
        self.input_path = input_path
        self.window_size = window_size
        self.future_steps = future_steps
        self.stock_safety_margin = stock_safety_margin
        self.batch_size = batch_size
        self.scaler_type = scaler_type
        self.threshold_method = threshold_method
        self.output_dir = output_dir
        self.show_plots = show_plots
        os.makedirs(self.output_dir, exist_ok=True)
        self.date_col = 'date'
        self.target_col = 'sales'
        self.feature_scaler = RobustScaler() if scaler_type == 'robust' else MinMaxScaler()
        self.target_scaler = RobustScaler() if scaler_type == 'robust' else MinMaxScaler()
        self.model = None
        self.time_series = None
        self.feature_cols = []
        self.logs = []
        self.training_history = []
        self.peak_threshold = None
        self.device = device
        self.model_path = f'{self.output_dir}/best_forecasting_lstm.pt'
        self.feature_scaler_path = f'{self.output_dir}/forecasting_feature_scaler.pkl'
        self.target_scaler_path = f'{self.output_dir}/forecasting_target_scaler.pkl'
        self.metadata_path = f'{self.output_dir}/forecasting_model_metadata.json'

    def log_action(self, action, status, details=None):
        self.logs.append({'timestamp': datetime.now().isoformat(), 'agent': 'ForecastingTrendAgent',
                          'action': action, 'status': status, 'details': details})

    def save_logs(self):
        with open(f'{self.output_dir}/forecasting_agent_logs.json', 'w') as f:
            json.dump(self.logs, f, indent=4, default=str)

    def load_data(self):
        df = pd.read_csv(self.input_path)
        df[self.date_col] = pd.to_datetime(df[self.date_col], errors='coerce')
        df[self.target_col] = pd.to_numeric(df[self.target_col], errors='coerce')
        df = df.dropna(subset=[self.date_col, self.target_col]).drop_duplicates().sort_values(self.date_col)
        if len(df) <= self.window_size + 10:
            raise ValueError('Not enough records for LSTM forecasting.')
        self.log_action('load_data', 'success', f'Loaded shape: {df.shape}')
        return df

    def aggregate_by_date(self, df):
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        agg_dict = {col: ('sum' if col == self.target_col else 'mean') for col in numeric_cols}
        self.time_series = (df.groupby(self.date_col).agg(agg_dict)
                            .reset_index().sort_values(self.date_col))
        self.log_action('aggregate_by_date', 'success', f'Shape: {self.time_series.shape}')
        return self.time_series

    def add_features(self, time_series):
        ts = time_series.copy()
        ts['day_of_week'] = ts[self.date_col].dt.dayofweek
        ts['month'] = ts[self.date_col].dt.month
        ts['day'] = ts[self.date_col].dt.day
        ts['is_weekend'] = ts['day_of_week'].isin([5, 6]).astype(int)
        ts['sin_week'] = np.sin(2 * np.pi * ts['day_of_week'] / 7)
        ts['cos_week'] = np.cos(2 * np.pi * ts['day_of_week'] / 7)
        ts['sin_month'] = np.sin(2 * np.pi * ts['month'] / 12)
        ts['cos_month'] = np.cos(2 * np.pi * ts['month'] / 12)
        for lag in [1, 7, 14, 30]:
            ts[f'sales_lag_{lag}'] = ts[self.target_col].shift(lag)
        for window in [7, 14, 30]:
            ts[f'sales_ma_{window}'] = ts[self.target_col].shift(1).rolling(window, min_periods=1).mean()
            ts[f'sales_std_{window}'] = ts[self.target_col].shift(1).rolling(window, min_periods=1).std()
        ts['sales_ema_7'] = ts[self.target_col].ewm(span=7, adjust=False).mean()
        ts['sales_ema_30'] = ts[self.target_col].ewm(span=30, adjust=False).mean()
        ts['sales_diff_1'] = ts[self.target_col].diff(1)
        ts['sales_pct_change'] = ts[self.target_col].pct_change()
        ts = ts.replace([np.inf, -np.inf], np.nan).fillna(0)
        self.time_series = ts
        self.log_action('feature_engineering', 'success', 'Features created')
        return ts

    def prepare_features_train(self, time_series):
        self.feature_cols = [col for col in time_series.columns
                             if col not in [self.date_col, self.target_col]
                             and pd.api.types.is_numeric_dtype(time_series[col])]
        X_raw = time_series[self.feature_cols].values
        y_raw = time_series[[self.target_col]].values
        cutoff = max(int(len(time_series) * 0.70), 1)
        self.feature_scaler.fit(X_raw[:cutoff])
        self.target_scaler.fit(y_raw[:cutoff])
        X_scaled = self.feature_scaler.transform(X_raw)
        y_scaled = self.target_scaler.transform(y_raw)
        return np.concatenate([y_scaled, X_scaled], axis=1), y_scaled

    def prepare_features_inference(self, time_series):
        for col in self.feature_cols:
            if col not in time_series.columns:
                time_series[col] = 0
        X_scaled = self.feature_scaler.transform(time_series[self.feature_cols].values)
        y_scaled = self.target_scaler.transform(time_series[[self.target_col]].values)
        return np.concatenate([y_scaled, X_scaled], axis=1), y_scaled

    def create_sequences(self, full_scaled, y_scaled, dates):
        X, y, seq_dates = [], [], []
        for i in range(len(full_scaled) - self.window_size):
            X.append(full_scaled[i:i + self.window_size])
            y.append(y_scaled[i + self.window_size])
            seq_dates.append(dates.iloc[i + self.window_size])
        return np.array(X), np.array(y), seq_dates

    def split_data(self, X, y, seq_dates):
        n = len(X)
        t, v = int(n * 0.70), int(n * 0.85)
        return X[:t], y[:t], X[t:v], y[t:v], X[v:], y[v:], seq_dates[v:]

    def train_model(self, X_train, y_train, X_val, y_val, epochs=35, lr=0.001, patience=10):
        Xtr = torch.tensor(X_train, dtype=torch.float32)
        ytr = torch.tensor(y_train, dtype=torch.float32)
        Xv = torch.tensor(X_val, dtype=torch.float32)
        yv = torch.tensor(y_val, dtype=torch.float32)
        loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=self.batch_size, shuffle=False)
        self.model = LSTMForecastingModel(input_size=Xtr.shape[2]).to(self.device)
        criterion = nn.HuberLoss(delta=1.0)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)
        best_val, best_state, patience_counter = np.inf, None, 0
        for epoch in range(epochs):
            self.model.train()
            tl = []
            for bx, by in loader:
                bx, by = bx.to(self.device), by.to(self.device)
                pred = self.model(bx)
                loss = criterion(pred, by)
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                tl.append(loss.item())
            avg_tl = float(np.mean(tl))
            self.model.eval()
            with torch.no_grad():
                vl = criterion(self.model(Xv.to(self.device)), yv.to(self.device)).item() if len(Xv) > 0 else avg_tl
            scheduler.step(vl)
            self.training_history.append({'epoch': epoch+1, 'train_loss': avg_tl, 'val_loss': float(vl)})
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f'  Epoch [{epoch+1}/{epochs}] Train: {avg_tl:.5f} | Val: {vl:.5f}')
            if vl < best_val:
                best_val = vl
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break
        if best_state:
            self.model.load_state_dict(best_state)
        torch.save(self.model.state_dict(), self.model_path)
        self.log_action('train_model', 'success', f'Saved to {self.model_path}')

    def inverse_target(self, values):
        return self.target_scaler.inverse_transform(values).flatten()

    def predict_real_values(self, X):
        self.model.eval()
        with torch.no_grad():
            pred = self.model(torch.tensor(X, dtype=torch.float32).to(self.device)).cpu().numpy()
        return np.clip(self.inverse_target(pred), 0, None)

    def compute_peak_threshold(self):
        hist = self.time_series[self.target_col].values
        mst = np.mean(hist) + np.std(hist)
        pt = np.percentile(hist, 90)
        threshold = max(mst, pt) if self.threshold_method == 'hybrid' else mst
        self.peak_threshold = threshold
        return threshold, self.threshold_method

    def evaluate_model(self, X_test, y_test, test_dates):
        y_pred = self.predict_real_values(X_test)
        y_true = self.inverse_target(y_test)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else 0.0
        smape = np.mean(2*np.abs(y_pred-y_true)/(np.abs(y_true)+np.abs(y_pred)+1e-8)) * 100
        threshold, method = self.compute_peak_threshold()
        metrics = {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2), 'SMAPE': float(smape),
                   'Peak_Threshold': float(threshold), 'Threshold_Method': method}
        test_df = pd.DataFrame({'date': test_dates, 'actual_sales': y_true, 'predicted_sales': y_pred,
                                'actual_peak': y_true >= threshold, 'predicted_peak': y_pred >= threshold})
        self.log_action('evaluate_model', 'success', metrics)
        return test_df, metrics

    def forecast_future(self, full_scaled):
        current_seq = full_scaled[-self.window_size:].copy()
        future_scaled = []
        self.model.eval()
        for _ in range(self.future_steps):
            inp = torch.tensor(current_seq, dtype=torch.float32).unsqueeze(0).to(self.device)
            with torch.no_grad():
                nxt = self.model(inp).cpu().numpy()[0]
            future_scaled.append(nxt)
            next_row = np.concatenate([nxt, current_seq[-1, 1:]])
            current_seq = np.vstack([current_seq[1:], next_row])
        return np.clip(self.inverse_target(np.array(future_scaled)), 0, None)

    def detect_future_peaks(self, future_sales):
        future_dates = pd.date_range(
            start=self.time_series[self.date_col].max() + pd.Timedelta(days=1),
            periods=self.future_steps, freq='D')
        future_df = pd.DataFrame({'date': future_dates, 'predicted_sales': future_sales})
        fpt = min(np.percentile(future_df['predicted_sales'], 90),
                  future_df['predicted_sales'].mean() + future_df['predicted_sales'].std())
        future_df['peak_threshold'] = fpt
        future_df['is_peak'] = future_df['predicted_sales'] >= fpt
        future_df['recommended_inventory'] = (future_df['predicted_sales'] * (1 + self.stock_safety_margin)).round()
        return future_df, future_df[future_df['is_peak']].copy()

    def analyze_trend(self, future_df):
        fv, lv = future_df['predicted_sales'].iloc[0], future_df['predicted_sales'].iloc[-1]
        change = lv - fv
        pct = (change / fv * 100) if fv != 0 else 0
        trend = 'Increasing demand' if pct > 5 else ('Decreasing demand' if pct < -5 else 'Stable demand')
        future_df['demand_change_pct_from_start'] = ((future_df['predicted_sales'] - fv) / fv * 100) if fv != 0 else 0
        return future_df, {'first_predicted_sales': float(fv), 'last_predicted_sales': float(lv),
                           'change': float(change), 'change_percentage': float(pct), 'trend': trend}

    def analyze_promotion_impact(self):
        if 'onpromotion' not in self.time_series.columns:
            return {'available': False, 'message': 'Promotion feature not found.'}
        corr = self.time_series['onpromotion'].corr(self.time_series[self.target_col])
        return {'available': True, 'correlation_onpromotion_sales': float(corr) if not pd.isna(corr) else None}

    def save_model_artifacts(self):
        with open(self.feature_scaler_path, 'wb') as f: pickle.dump(self.feature_scaler, f)
        with open(self.target_scaler_path, 'wb') as f: pickle.dump(self.target_scaler, f)
        metadata = {'feature_cols': self.feature_cols, 'window_size': self.window_size,
                    'future_steps': self.future_steps, 'stock_safety_margin': self.stock_safety_margin,
                    'scaler_type': self.scaler_type, 'threshold_method': self.threshold_method,
                    'input_size': len(self.feature_cols)+1, 'model_path': self.model_path}
        with open(self.metadata_path, 'w') as f: json.dump(metadata, f, indent=4)

    def load_model_artifacts(self):
        with open(self.metadata_path) as f: metadata = json.load(f)
        self.feature_cols = metadata['feature_cols']
        with open(self.feature_scaler_path, 'rb') as f: self.feature_scaler = pickle.load(f)
        with open(self.target_scaler_path, 'rb') as f: self.target_scaler = pickle.load(f)
        self.model = LSTMForecastingModel(input_size=metadata['input_size']).to(self.device)
        self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
        self.model.eval()

    def save_outputs(self, future_df, peaks, test_df, summary):
        future_df.to_csv(f'{self.output_dir}/forecasting_results.csv', index=False)
        peaks.to_csv(f'{self.output_dir}/detected_peak_periods.csv', index=False)
        if test_df is not None:
            test_df.to_csv(f'{self.output_dir}/test_actual_vs_predicted.csv', index=False)
        with open(f'{self.output_dir}/forecasting_summary.json', 'w') as f:
            json.dump(summary, f, indent=4, default=str)
        self.save_logs()

    def save_training_metrics(self, metrics):
        with open(f'{self.output_dir}/forecasting_training_metrics.json', 'w') as f:
            json.dump(metrics, f, indent=4, default=str)

    def load_training_metrics(self):
        path = f'{self.output_dir}/forecasting_training_metrics.json'
        if not os.path.exists(path): return {}
        with open(path) as f: return json.load(f)

    def save_plot(self, filename):
        plt.tight_layout()
        plt.savefig(f'{self.output_dir}/{filename}')
        plt.close()

    def plot_forecast(self, future_df, peaks):
        plt.figure(figsize=(12, 5))
        plt.plot(future_df['date'], future_df['predicted_sales'], marker='o', label='Future predicted sales')
        plt.axhline(y=future_df['peak_threshold'].iloc[0], linestyle='--', label='Peak threshold')
        if len(peaks) > 0:
            plt.scatter(peaks['date'], peaks['predicted_sales'], s=80, label='Detected peaks', color='red')
        plt.title('Future Demand Forecasting & Peak Detection')
        plt.xlabel('Date'); plt.ylabel('Sales'); plt.xticks(rotation=45); plt.legend()
        self.save_plot('forecasting_future_peaks.png')

    def plot_training_history(self):
        if not self.training_history: return
        hdf = pd.DataFrame(self.training_history)
        hdf.to_csv(f'{self.output_dir}/forecasting_training_history.csv', index=False)
        plt.figure(figsize=(10, 5))
        plt.plot(hdf['epoch'], hdf['train_loss'], label='Train Loss')
        plt.plot(hdf['epoch'], hdf['val_loss'], label='Validation Loss')
        plt.title('Forecasting LSTM - Training vs Validation Loss')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
        self.save_plot('forecasting_training_loss.png')

    def run_train(self, epochs=35, lr=0.001):
        df = self.load_data()
        ts = self.aggregate_by_date(df)
        ts = self.add_features(ts)
        full_scaled, y_scaled = self.prepare_features_train(ts)
        X, y, seq_dates = self.create_sequences(full_scaled, y_scaled, ts[self.date_col])
        Xtr, ytr, Xv, yv, Xte, yte, te_dates = self.split_data(X, y, seq_dates)
        self.train_model(Xtr, ytr, Xv, yv, epochs=epochs, lr=lr)
        test_df, metrics = self.evaluate_model(Xte, yte, te_dates)
        self.save_training_metrics(metrics)
        future_sales = self.forecast_future(full_scaled)
        future_df, peaks = self.detect_future_peaks(future_sales)
        future_df, trend = self.analyze_trend(future_df)
        promo = self.analyze_promotion_impact()
        inv = {'safety_margin': self.stock_safety_margin,
               'total_predicted_demand': float(future_df['predicted_sales'].sum()),
               'total_recommended_inventory': float(future_df['recommended_inventory'].sum())}
        summary = {'mode': 'train', 'metrics': metrics, 'trend_analysis': trend,
                   'promotion_impact': promo, 'inventory_requirements': inv,
                   'number_of_detected_future_peaks': int(len(peaks)),
                   'model_type': 'LSTM Forecasting Model', 'model_saved_path': self.model_path}
        self.save_model_artifacts()
        self.save_outputs(future_df, peaks, test_df, summary)
        self.plot_forecast(future_df, peaks)
        self.plot_training_history()
        self.log_action('run_train', 'success', 'Training completed')
        self.save_logs()
        return future_df, peaks, summary

    def run_inference(self):
        self.load_model_artifacts()
        df = self.load_data()
        ts = self.aggregate_by_date(df)
        ts = self.add_features(ts)
        full_scaled, _ = self.prepare_features_inference(ts)
        future_sales = self.forecast_future(full_scaled)
        future_df, peaks = self.detect_future_peaks(future_sales)
        future_df, trend = self.analyze_trend(future_df)
        promo = self.analyze_promotion_impact()
        inv = {'safety_margin': self.stock_safety_margin,
               'total_predicted_demand': float(future_df['predicted_sales'].sum()),
               'total_recommended_inventory': float(future_df['recommended_inventory'].sum())}
        metrics = self.load_training_metrics()
        summary = {'mode': 'inference', 'metrics': metrics, 'trend_analysis': trend,
                   'promotion_impact': promo, 'inventory_requirements': inv,
                   'number_of_detected_future_peaks': int(len(peaks)),
                   'model_type': 'Loaded LSTM Forecasting Model', 'model_loaded_path': self.model_path}
        self.save_outputs(future_df, peaks, None, summary)
        self.plot_forecast(future_df, peaks)
        self.log_action('run_inference', 'success', 'Inference completed')
        self.save_logs()
        return future_df, peaks, summary

print('✅ ForecastingTrendAgent defined')

In [ ]:
# ════════════════════════════════════════════════════════════════
# LSTM AUTOENCODER + ANOMALY DETECTION AGENT
# ════════════════════════════════════════════════════════════════

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size=96, latent_size=32, num_layers=1, dropout_rate=0.2):
        super().__init__()
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.latent = nn.Linear(hidden_size, latent_size)
        self.decoder_input = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        enc, _ = self.encoder(x)
        latent = self.dropout(self.latent(enc[:, -1, :]))
        repeated = self.decoder_input(latent).unsqueeze(1).repeat(1, x.shape[1], 1)
        dec, _ = self.decoder(repeated)
        return self.output_layer(dec)


class DLAnomalyDetectionAgent:
    def __init__(self, input_path, window_size=30, batch_size=50, anomaly_percentile=95,
                 z_threshold=2.8, promotion_threshold=0.2, dropout_rate=0.2,
                 output_dir='outputs', show_plots=False):
        self.input_path = input_path
        self.window_size = window_size
        self.batch_size = batch_size
        self.anomaly_percentile = anomaly_percentile
        self.z_threshold = z_threshold
        self.promotion_threshold = promotion_threshold
        self.dropout_rate = dropout_rate
        self.output_dir = output_dir
        self.show_plots = show_plots
        os.makedirs(self.output_dir, exist_ok=True)
        self.date_col = 'date'
        self.target_col = 'sales'
        self.scaler = MinMaxScaler()
        self.logs = []; self.training_history = []
        self.model = None; self.df = None; self.daily_df = None
        self.results = None; self.anomalies = None; self.feature_cols = []
        self.model_path = f'{self.output_dir}/best_anomaly_lstm_autoencoder.pt'
        self.scaler_path = f'{self.output_dir}/anomaly_scaler.pkl'
        self.metadata_path = f'{self.output_dir}/anomaly_model_metadata.json'

    def log_action(self, action, status, details=None):
        self.logs.append({'timestamp': datetime.now().isoformat(), 'agent': 'DLAnomalyDetectionAgent',
                          'action': action, 'status': status, 'details': details})

    def save_logs(self):
        with open(f'{self.output_dir}/anomaly_agent_logs.json', 'w') as f:
            json.dump(self.logs, f, indent=4, default=str)

    def load_data(self):
        df = pd.read_csv(self.input_path)
        df[self.date_col] = pd.to_datetime(df[self.date_col], errors='coerce')
        df[self.target_col] = pd.to_numeric(df[self.target_col], errors='coerce')
        df = df.dropna(subset=[self.date_col, self.target_col]).sort_values(self.date_col)
        if len(df) <= self.window_size + 10:
            raise ValueError('Not enough records for anomaly detection.')
        self.df = df
        self.log_action('load_data', 'success', f'Loaded shape: {df.shape}')
        return df

    def aggregate_by_date(self):
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        agg_dict = {col: ('sum' if col == self.target_col else 'mean') for col in numeric_cols}
        self.daily_df = (self.df.groupby(self.date_col).agg(agg_dict).reset_index().sort_values(self.date_col))
        self.log_action('aggregate_by_date', 'success', f'Shape: {self.daily_df.shape}')
        return self.daily_df

    def create_features(self):
        df = self.daily_df.copy()
        df['sales_lag_1'] = df[self.target_col].shift(1)
        df['sales_lag_7'] = df[self.target_col].shift(7)
        df['sales_diff'] = df[self.target_col] - df['sales_lag_1']
        df['sales_pct_change'] = df[self.target_col].pct_change()
        df['rolling_mean_7'] = df[self.target_col].rolling(7, min_periods=1).mean()
        df['rolling_std_7'] = df[self.target_col].rolling(7, min_periods=1).std()
        df['z_score'] = ((df[self.target_col] - df['rolling_mean_7']) /
                         df['rolling_std_7'].replace(0, np.nan))
        df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
        self.daily_df = df
        self.log_action('create_features', 'success', 'Anomaly features created')
        return df

    def select_model_features_train(self):
        preferred = ['sales', 'onpromotion', 'transactions', 'dcoilwtico', 'day_of_week',
                     'is_weekend', 'sales_lag_1', 'sales_lag_7', 'sales_diff',
                     'sales_pct_change', 'rolling_mean_7', 'rolling_std_7', 'z_score']
        self.feature_cols = [c for c in preferred if c in self.daily_df.columns
                             and pd.api.types.is_numeric_dtype(self.daily_df[c])]
        if not self.feature_cols:
            raise ValueError('No numeric features available.')
        self.log_action('select_model_features_train', 'success', self.feature_cols)
        return self.feature_cols

    def select_model_features_inference(self):
        for col in self.feature_cols:
            if col not in self.daily_df.columns:
                self.daily_df[col] = 0
        return self.feature_cols

    def create_sequences_from_scaled(self, scaled_values):
        X, seq_dates, normal_mask = [], [], []
        for i in range(len(scaled_values) - self.window_size):
            X.append(scaled_values[i:i + self.window_size])
            row = self.daily_df.iloc[i + self.window_size]
            seq_dates.append(row[self.date_col])
            normal_mask.append(abs(row['z_score']) < self.z_threshold and
                               abs(row['sales_pct_change']) < 0.5)
        X = np.array(X)
        if len(X) == 0:
            raise ValueError('No sequences created. Reduce window_size.')
        self.sequence_dates = seq_dates
        self.normal_sequence_mask = np.array(normal_mask)
        self.log_action('prepare_sequences', 'success', f'Created sequences: {X.shape}')
        return X

    def prepare_sequences_train(self):
        self.select_model_features_train()
        scaled = self.scaler.fit_transform(self.daily_df[self.feature_cols].values)
        return self.create_sequences_from_scaled(scaled)

    def prepare_sequences_inference(self):
        self.select_model_features_inference()
        scaled = self.scaler.transform(self.daily_df[self.feature_cols].values)
        return self.create_sequences_from_scaled(scaled)

    def train_autoencoder(self, X, epochs=50, lr=0.001, patience=8, min_delta=0.0001):
        Xt = torch.tensor(X, dtype=torch.float32)
        normal_X = Xt[self.normal_sequence_mask] if len(Xt[self.normal_sequence_mask]) >= 100 else Xt
        ts = max(int(len(normal_X) * 0.8), 1)
        Xtr, Xv = normal_X[:ts], normal_X[ts:]
        if len(Xv) == 0: Xv = Xtr
        loader = DataLoader(TensorDataset(Xtr, Xtr), batch_size=self.batch_size, shuffle=False)
        vloader = DataLoader(TensorDataset(Xv, Xv), batch_size=self.batch_size, shuffle=False)
        self.model = LSTMAutoencoder(input_size=X.shape[2], dropout_rate=self.dropout_rate)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        best_val, best_state, patience_counter = float('inf'), None, 0
        for epoch in range(epochs):
            self.model.train()
            tl = []
            for bx, _ in loader:
                rec = self.model(bx)
                loss = criterion(rec, bx)
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                tl.append(loss.item())
            avg_tl = float(np.mean(tl))
            self.model.eval()
            vl = []
            with torch.no_grad():
                for bx, _ in vloader:
                    vl.append(criterion(self.model(bx), bx).item())
            avg_vl = float(np.mean(vl))
            self.training_history.append({'epoch': epoch+1, 'train_loss': avg_tl, 'val_loss': avg_vl})
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f'  Epoch [{epoch+1}/{epochs}] Train: {avg_tl:.6f} | Val: {avg_vl:.6f}')
            if avg_vl < best_val - min_delta:
                best_val = avg_vl
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break
        if best_state: self.model.load_state_dict(best_state)
        self.final_loss = float(avg_tl)
        self.best_validation_loss = float(best_val)
        torch.save(self.model.state_dict(), self.model_path)
        self.log_action('train_autoencoder', 'success', f'Saved to {self.model_path}')

    def compute_reconstruction_errors(self, X, use_training_threshold=True):
        self.model.eval()
        with torch.no_grad():
            rec = self.model(torch.tensor(X, dtype=torch.float32)).numpy()
        errors = np.mean((X - rec) ** 2, axis=(1, 2))
        if use_training_threshold:
            normal_errors = errors[self.normal_sequence_mask] if len(errors[self.normal_sequence_mask]) >= 100 else errors
            threshold = np.percentile(normal_errors, self.anomaly_percentile)
            self.anomaly_threshold = float(threshold)
        else:
            saved = getattr(self, 'anomaly_threshold', None)
            adaptive = np.percentile(errors, 95)
            threshold = max(saved, adaptive) if saved else adaptive
        return pd.DataFrame({'date': self.sequence_dates, 'reconstruction_error': errors,
                             'anomaly_threshold': threshold,
                             'autoencoder_anomaly': errors > threshold})

    def enrich_anomaly_results(self, results):
        enriched = results.merge(self.daily_df.copy(), on='date', how='left')
        enriched['sales_spike'] = enriched['z_score'] >= self.z_threshold
        enriched['sales_drop'] = enriched['z_score'] <= -self.z_threshold
        if 'onpromotion' in enriched.columns:
            enriched['promotion_anomaly'] = ((enriched['onpromotion'] > 0) &
                                             (enriched['sales_pct_change'] < -self.promotion_threshold))
        else:
            enriched['promotion_anomaly'] = False
        enriched['is_anomaly'] = (enriched['autoencoder_anomaly'] | enriched['sales_spike'] |
                                  enriched['sales_drop'] | enriched['promotion_anomaly'])

        def anomaly_type(row):
            if not row['is_anomaly']: return 'Normal'
            if row['sales_spike']: return 'Sudden Sales Spike'
            if row['sales_drop']: return 'Sudden Sales Drop'
            if row['promotion_anomaly']: return 'Suspicious Promotional Behavior'
            return 'Inconsistent Demand Pattern'

        def risk_level(row):
            if not row['is_anomaly']: return 'Normal'
            score = int(row['autoencoder_anomaly']) + int(row['sales_spike'] or row['sales_drop']) + int(row['promotion_anomaly'])
            return 'High Risk' if score >= 3 else ('Medium Risk' if score == 2 else 'Low Risk')

        enriched['anomaly_type'] = enriched.apply(anomaly_type, axis=1)
        enriched['risk_level'] = enriched.apply(risk_level, axis=1)
        self.results = enriched
        self.anomalies = enriched[enriched['is_anomaly']].copy()
        self.log_action('enrich_anomaly_results', 'success',
                        {'total_results': len(self.results), 'total_anomalies': len(self.anomalies)})
        return self.results, self.anomalies

    def generate_summary(self, mode):
        rate = float(len(self.anomalies)/len(self.results)) if len(self.results) > 0 else 0.0
        cols = [c for c in ['date', 'sales', 'reconstruction_error', 'risk_level', 'anomaly_type'] if c in self.anomalies.columns]
        return {'mode': mode,
                'training_metrics': {'final_reconstruction_loss': getattr(self, 'final_loss', None),
                                     'best_validation_loss': getattr(self, 'best_validation_loss', None),
                                     'mean_reconstruction_error': float(self.results['reconstruction_error'].mean()),
                                     'anomaly_threshold': float(self.results['anomaly_threshold'].iloc[0]),
                                     'features_used': self.feature_cols},
                'total_records_analyzed': int(len(self.results)),
                'total_anomalies_detected': int(len(self.anomalies)),
                'anomaly_rate': rate,
                'risk_distribution': self.anomalies['risk_level'].value_counts().to_dict(),
                'anomaly_type_distribution': self.anomalies['anomaly_type'].value_counts().to_dict(),
                'highest_risk_dates': self.anomalies[
                    self.anomalies['risk_level'].isin(['High Risk', 'Medium Risk'])][cols].head(10).to_dict(orient='records')}

    def save_model_artifacts(self):
        with open(self.scaler_path, 'wb') as f: pickle.dump(self.scaler, f)
        metadata = {'feature_cols': self.feature_cols, 'window_size': self.window_size,
                    'input_size': len(self.feature_cols), 'anomaly_threshold': self.anomaly_threshold,
                    'anomaly_percentile': self.anomaly_percentile, 'z_threshold': self.z_threshold,
                    'dropout_rate': self.dropout_rate, 'model_path': self.model_path,
                    'final_reconstruction_loss': getattr(self, 'final_loss', None),
                    'best_validation_loss': getattr(self, 'best_validation_loss', None)}
        with open(self.metadata_path, 'w') as f: json.dump(metadata, f, indent=4)

    def load_model_artifacts(self):
        with open(self.metadata_path) as f: metadata = json.load(f)
        self.feature_cols = metadata['feature_cols']
        self.anomaly_threshold = metadata['anomaly_threshold']
        self.final_loss = metadata.get('final_reconstruction_loss')
        self.best_validation_loss = metadata.get('best_validation_loss')
        with open(self.scaler_path, 'rb') as f: self.scaler = pickle.load(f)
        self.model = LSTMAutoencoder(input_size=metadata['input_size'], dropout_rate=metadata['dropout_rate'])
        self.model.load_state_dict(torch.load(self.model_path, map_location='cpu'))
        self.model.eval()

    def save_outputs(self, summary):
        self.results.to_csv(f'{self.output_dir}/anomaly_detection_results.csv', index=False)
        self.anomalies.to_csv(f'{self.output_dir}/detected_anomalies.csv', index=False)
        with open(f'{self.output_dir}/anomaly_summary.json', 'w') as f:
            json.dump(summary, f, indent=4, default=str)
        with open(f'{self.output_dir}/anomaly_training_history.json', 'w') as f:
            json.dump(self.training_history, f, indent=4)
        self.save_logs()

    def save_plot(self, filename):
        plt.tight_layout()
        plt.savefig(f'{self.output_dir}/{filename}')
        plt.close()

    def plot_anomalies(self):
        plt.figure(figsize=(12, 5))
        plt.plot(self.daily_df['date'], self.daily_df[self.target_col], label='Sales')
        if len(self.anomalies) > 0:
            plt.scatter(self.anomalies['date'], self.anomalies[self.target_col],
                        color='red', s=60, label='Detected anomalies')
        plt.title('Sales Anomaly Detection using LSTM Autoencoder')
        plt.xlabel('Date'); plt.ylabel('Sales'); plt.xticks(rotation=45); plt.legend()
        self.save_plot('anomaly_detected_points.png')

    def plot_training_history(self):
        if not self.training_history: return
        hdf = pd.DataFrame(self.training_history)
        hdf.to_csv(f'{self.output_dir}/anomaly_training_history.csv', index=False)
        plt.figure(figsize=(10, 5))
        plt.plot(hdf['epoch'], hdf['train_loss'], label='Train Loss')
        plt.plot(hdf['epoch'], hdf['val_loss'], label='Validation Loss')
        plt.title('Anomaly LSTM Autoencoder - Training vs Validation Loss')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
        self.save_plot('anomaly_training_loss.png')

    def run_train(self, epochs=50, lr=0.001):
        self.load_data(); self.aggregate_by_date(); self.create_features()
        X = self.prepare_sequences_train()
        self.train_autoencoder(X, epochs=epochs, lr=lr)
        raw = self.compute_reconstruction_errors(X, use_training_threshold=True)
        self.enrich_anomaly_results(raw)
        summary = self.generate_summary('train')
        self.save_model_artifacts(); self.save_outputs(summary)
        self.plot_anomalies(); self.plot_training_history()
        self.log_action('run_train', 'success', 'Training completed'); self.save_logs()
        return self.results, self.anomalies, summary

    def run_inference(self):
        self.load_model_artifacts(); self.load_data(); self.aggregate_by_date(); self.create_features()
        X = self.prepare_sequences_inference()
        raw = self.compute_reconstruction_errors(X, use_training_threshold=False)
        self.enrich_anomaly_results(raw)
        summary = self.generate_summary('inference')
        self.save_outputs(summary); self.plot_anomalies()
        self.log_action('run_inference', 'success', 'Inference completed'); self.save_logs()
        return self.results, self.anomalies, summary

print('✅ DLAnomalyDetectionAgent defined')

In [ ]:
# ════════════════════════════════════════════════════════════════
# RECOMMENDATION, HUMAN-IN-THE-LOOP & REPORT AGENTS
# ════════════════════════════════════════════════════════════════

class RecommendationAgent:
    def __init__(self, mode, future_df=None, peaks=None, forecasting_summary=None,
                 detected_anomalies=None, anomaly_summary=None):
        self.mode = mode
        self.future_df = future_df; self.peaks = peaks
        self.forecasting_summary = forecasting_summary or {}
        self.detected_anomalies = detected_anomalies
        self.anomaly_summary = anomaly_summary or {}
        self.recommendations = []; self.logs = []

    def log_action(self, action, status, details=None):
        self.logs.append({'timestamp': datetime.now().isoformat(), 'agent': 'RecommendationAgent',
                          'action': action, 'status': status, 'details': details})

    def add_recommendation(self, category, priority, recommendation, reason):
        self.recommendations.append({'category': category, 'priority': priority,
                                     'recommendation': recommendation, 'reason': reason})

    def generate_forecasting_recommendations(self):
        trend = self.forecasting_summary['trend_analysis']['trend']
        pct = self.forecasting_summary['trend_analysis']['change_percentage']
        if trend == 'Increasing demand':
            self.add_recommendation('Demand Forecasting', 'High',
                'Increase stock availability and prepare additional inventory.',
                f'The model predicts a demand increase of {pct:.2f}%.')
        elif trend == 'Decreasing demand':
            self.add_recommendation('Demand Forecasting', 'Medium',
                'Reduce excessive stock preparation for the next period.',
                f'The model predicts a demand decrease of {pct:.2f}%.')
        else:
            self.add_recommendation('Demand Forecasting', 'Low',
                'Maintain current stock planning strategy.', 'Demand is predicted to be stable.')
        if self.peaks is not None and len(self.peaks) > 0:
            self.add_recommendation('Peak Demand', 'High',
                'Prepare additional inventory before detected peak demand dates.',
                f'Future peak periods: {self.peaks["date"].astype(str).tolist()}.')
        inv = self.forecasting_summary['inventory_requirements']
        self.add_recommendation('Inventory Management', 'High',
            f'Prepare inventory with {int(inv["safety_margin"]*100)}% safety margin.',
            f'Expected demand: ~{inv["total_predicted_demand"]:.0f} units.')

    def generate_anomaly_recommendations(self):
        rate = self.anomaly_summary['anomaly_rate']
        total = self.anomaly_summary['total_anomalies_detected']
        risk_dist = self.anomaly_summary['risk_distribution']
        types = self.anomaly_summary['anomaly_type_distribution']
        priority = 'High' if rate > 0.10 else 'Medium'
        self.add_recommendation('Anomaly Monitoring', priority,
            'Investigate abnormal sales and promotion behavior before final stock decisions.',
            f'{total} anomalies detected. Risk distribution: {risk_dist}.')
        self.add_recommendation('Operational Risk', priority,
            'Review the main anomaly categories to identify operational inconsistencies.',
            f'Detected anomaly types: {types}.')
        self.add_recommendation('Business Control', 'High',
            'Validate suspicious periods before taking inventory or promotion decisions.',
            'Human review required — abnormal behavior may indicate stock or promotion issues.')

    def generate_business_insight(self):
        # Rule-based insight (replaces Ollama/CrewAI for Colab compatibility)
        if self.anomaly_summary and self.anomaly_summary.get('total_anomalies_detected') is not None:
            total = self.anomaly_summary.get('total_anomalies_detected', 'N/A')
            records = self.anomaly_summary.get('total_records_analyzed', 'N/A')
            rate = self.anomaly_summary.get('anomaly_rate', 0)
            risk = self.anomaly_summary.get('risk_distribution', {})
            types = self.anomaly_summary.get('anomaly_type_distribution', {})
            main_type = max(types, key=types.get) if types else 'unknown'
            actions = [r['recommendation'] for r in self.recommendations[:3]]
            return (f'SUMMARY:\n- {total} anomalies detected out of {records} records.\n'
                    f'KEY FINDINGS:\n- Anomaly rate: {rate*100:.2f}%. Main type: {main_type}. Risk: {risk}.\n'
                    f'RECOMMENDED ACTIONS:\n' + '\n'.join(f'- {a}' for a in actions))
        if self.forecasting_summary and self.forecasting_summary.get('trend_analysis'):
            trend = self.forecasting_summary['trend_analysis']
            inv = self.forecasting_summary.get('inventory_requirements', {})
            peaks_n = self.forecasting_summary.get('number_of_detected_future_peaks', 0)
            actions = [r['recommendation'] for r in self.recommendations[:3]]
            return (f'SUMMARY:\n- Forecast predicts {trend["trend"]} with {peaks_n} future peak periods.\n'
                    f'KEY FINDINGS:\n- Demand change: {trend["change_percentage"]:.2f}%.\n'
                    f'- Total predicted demand: {inv.get("total_predicted_demand", "N/A"):.0f} units.\n'
                    f'RECOMMENDED ACTIONS:\n' + '\n'.join(f'- {a}' for a in actions))
        return 'No valid results available for insight generation.'

    def generate_action_plan(self):
        if self.mode == 'forecasting':
            return {'stock_strategy': 'Adjust stock based on forecasted demand and peak periods.',
                    'promotion_strategy': 'Use promotion impact analysis for inventory campaigns.',
                    'decision_strategy': 'Send forecasting recommendations for human validation.'}
        return {'risk_strategy': 'Review detected anomalies before approving operational decisions.',
                'control_strategy': 'Investigate suspicious periods and abnormal demand patterns.',
                'decision_strategy': 'Send anomaly recommendations for human validation.'}

    def run(self, output_path='outputs/recommendation_report.json'):
        if self.mode == 'forecasting':
            self.generate_forecasting_recommendations()
        else:
            self.generate_anomaly_recommendations()
        insight = self.generate_business_insight()
        self.log_action('run', 'success', f'Recommendations generated for mode: {self.mode}')
        output = {'mode': self.mode, 'recommendations': self.recommendations,
                  'business_insight': insight, 'action_plan': self.generate_action_plan(), 'logs': self.logs}
        with open(output_path, 'w') as f:
            json.dump(output, f, indent=4, default=str)
        return output


class HumanInLoopAgent:
    def __init__(self, recommendation_output, output_path='outputs/human_validation_report.json'):
        self.recommendation_output = recommendation_output
        self.recommendations = recommendation_output.get('recommendations', [])
        self.output_path = output_path
        self.validation_results = []; self.logs = []

    def log_action(self, action, status, details=None):
        self.logs.append({'timestamp': datetime.now().isoformat(), 'agent': 'HumanInLoopAgent',
                          'action': action, 'status': status, 'details': details})

    def auto_validate(self, rec):
        priority = rec.get('priority', 'Low')
        comment = {'High': 'High-priority — requires careful review before implementation.',
                   'Medium': 'Accepted with moderate business impact.',
                   'Low': 'Low-risk recommendation automatically accepted.'}.get(priority, '')
        return {'category': rec.get('category', '—'), 'priority': priority,
                'recommendation': rec.get('recommendation', '—'),
                'approval_status': 'Approved', 'reviewer_comment': comment}

    def run(self):
        self.validation_results = [self.auto_validate(r) for r in self.recommendations]
        approved = sum(1 for r in self.validation_results if r['approval_status'] == 'Approved')
        rejected = sum(1 for r in self.validation_results if r['approval_status'] == 'Rejected')
        needs = sum(1 for r in self.validation_results if r['approval_status'] == 'Needs Review')
        status = 'Validated' if rejected == 0 and needs == 0 else ('Partially Validated' if approved > 0 else 'Rejected')
        summary = {'total_recommendations': len(self.validation_results), 'approved': approved,
                   'rejected': rejected, 'needs_review': needs, 'final_status': status}
        output = {'validation_results': self.validation_results, 'validation_summary': summary, 'logs': self.logs}
        with open(self.output_path, 'w') as f:
            json.dump(output, f, indent=4)
        self.log_action('run', 'success', f'Auto-validated {len(self.validation_results)} recommendations')
        return output


class ReportGenerationAgent:
    def __init__(self, mode, forecasting_summary=None, anomaly_summary=None,
                 recommendation_output=None, human_validation_output=None, output_dir='outputs'):
        self.mode = mode
        self.forecasting_summary = forecasting_summary or {}
        self.anomaly_summary = anomaly_summary or {}
        self.recommendation_output = recommendation_output or {}
        self.human_validation_output = human_validation_output or {}
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def sf(self, v, d=2, default='N/A'):
        try:
            v = float(v)
            return default if (math.isnan(v) or math.isinf(v)) else f'{v:.{d}f}'
        except: return default

    def run(self):
        recs = self.recommendation_output.get('recommendations', [])
        val = self.human_validation_output.get('validation_summary', {})
        insight = self.recommendation_output.get('business_insight', '')

        if self.mode == 'forecasting':
            trend = self.forecasting_summary.get('trend_analysis', {})
            metrics = self.forecasting_summary.get('metrics', {})
            inv = self.forecasting_summary.get('inventory_requirements', {})
            peaks_n = self.forecasting_summary.get('number_of_detected_future_peaks', 0)
            report_data = {
                'analysis_mode': 'forecasting', 'generated_at': datetime.now().isoformat(),
                'model_type': self.forecasting_summary.get('model_type', 'LSTM Forecasting'),
                'forecasting_results': {
                    'trend': trend.get('trend', 'N/A'),
                    'change_percentage': self.sf(trend.get('change_percentage', 0)),
                    'detected_future_peaks': peaks_n,
                    'total_predicted_demand': self.sf(inv.get('total_predicted_demand', 0), 0),
                    'recommended_inventory': self.sf(inv.get('total_recommended_inventory', 0), 0),
                    'metrics': {k: self.sf(v) for k, v in metrics.items() if isinstance(v, (int, float))}},
                'business_recommendations': recs,
                'business_insight': insight,
                'human_validation': val}
        else:
            report_data = {
                'analysis_mode': 'anomaly', 'generated_at': datetime.now().isoformat(),
                'anomaly_detection_results': {
                    'total_records': self.anomaly_summary.get('total_records_analyzed', 0),
                    'total_anomalies': self.anomaly_summary.get('total_anomalies_detected', 0),
                    'anomaly_rate': self.sf(self.anomaly_summary.get('anomaly_rate', 0), 4),
                    'risk_distribution': self.anomaly_summary.get('risk_distribution', {}),
                    'anomaly_type_distribution': self.anomaly_summary.get('anomaly_type_distribution', {})},
                'business_recommendations': recs,
                'business_insight': insight,
                'human_validation': val}

        with open(f'{self.output_dir}/final_business_report.json', 'w') as f:
            json.dump(report_data, f, indent=4, default=str)

        # Plain text report
        lines = [f'=== RETAIL ANALYTICS BUSINESS REPORT ===',
                 f'Generated: {report_data["generated_at"]}',
                 f'Analysis Mode: {self.mode.upper()}', '']
        if self.mode == 'forecasting':
            fr = report_data['forecasting_results']
            lines += [f'DEMAND TREND: {fr["trend"]}',
                      f'Change: {fr["change_percentage"]}% | Peak Periods: {fr["detected_future_peaks"]}',
                      f'Predicted Demand: {fr["total_predicted_demand"]} units',
                      f'Recommended Inventory: {fr["recommended_inventory"]} units', '']
            for k, v in fr['metrics'].items():
                lines.append(f'  {k}: {v}')
        else:
            ar = report_data['anomaly_detection_results']
            lines += [f'Total Records: {ar["total_records"]}',
                      f'Anomalies Detected: {ar["total_anomalies"]} ({ar["anomaly_rate"]})',
                      f'Risk Distribution: {ar["risk_distribution"]}', '']
        lines += ['', 'BUSINESS RECOMMENDATIONS:']
        for r in recs:
            lines.append(f'  [{r["priority"]}] {r["category"]}: {r["recommendation"]}')
        lines += ['', 'BUSINESS INSIGHT:', insight, '',
                  f'VALIDATION STATUS: {val.get("final_status", "N/A")}']
        txt = '\n'.join(lines)
        with open(f'{self.output_dir}/final_business_report.txt', 'w') as f:
            f.write(txt)
        return report_data

print('✅ RecommendationAgent, HumanInLoopAgent & ReportGenerationAgent defined')

## 🔧 Section 4 – Configure & Run Preprocessing

In [ ]:
# ── Pipeline settings ─────────────────────────────────────────────────────────
# DATA_PATH is already set above from the upload step
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('🔄 Running Preprocessing Agent...')
prep_agent = RetailPreprocessingAgent(input_path=DATA_PATH)
clean_df, model_df = prep_agent.run(
    clean_output_path=f'{OUTPUT_DIR}/clean_merged_data.csv',
    model_output_path=f'{OUTPUT_DIR}/model_ready_data.csv'
)

if clean_df is not None:
    print(f'\n✅ Preprocessing complete!')
    print(f'   Main file   : {prep_agent.main_file}')
    print(f'   Date column : {prep_agent.date_col}')
    print(f'   Target col  : {prep_agent.target_col}')
    print(f'   Clean shape : {clean_df.shape}')
    print(f'   Model-ready : {model_df.shape}')
    print(f'\nModel-ready data sample:')
    display(model_df.head())
else:
    print('❌ Preprocessing failed. Check the error above.')

## 🏋️ Section 5 – Train Models

Choose which model(s) to train. Training runs on GPU if available.

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
WINDOW_SIZE   = 30
FUTURE_STEPS  = 30
BATCH_SIZE    = 50
FORECAST_EPOCHS = 35
ANOMALY_EPOCHS  = 50
LR = 0.001

print(f'Device: {device}')
print(f'Window size: {WINDOW_SIZE} | Future steps: {FUTURE_STEPS}')

In [ ]:
# ── Train LSTM Forecasting Model ─────────────────────────────────────────────
print('=' * 55)
print('  Training LSTM Forecasting Model')
print('=' * 55)

forecast_agent = ForecastingTrendAgent(
    input_path=f'{OUTPUT_DIR}/model_ready_data.csv',
    window_size=WINDOW_SIZE,
    future_steps=FUTURE_STEPS,
    stock_safety_margin=0.15,
    batch_size=BATCH_SIZE,
    scaler_type='robust',
    threshold_method='hybrid',
    output_dir=OUTPUT_DIR,
    show_plots=False
)

future_df, peaks, forecasting_summary = forecast_agent.run_train(
    epochs=FORECAST_EPOCHS, lr=LR
)

print('\n✅ Forecasting model trained!')
print(f'   Trend        : {forecasting_summary["trend_analysis"]["trend"]}')
print(f'   Change %     : {forecasting_summary["trend_analysis"]["change_percentage"]:.2f}%')
print(f'   Peak periods : {forecasting_summary["number_of_detected_future_peaks"]}')
m = forecasting_summary['metrics']
print(f'   MAE={m["MAE"]:.2f} | RMSE={m["RMSE"]:.2f} | R²={m["R2"]:.4f} | SMAPE={m["SMAPE"]:.2f}%')

In [ ]:
# ── Train LSTM Autoencoder (Anomaly Detection) ────────────────────────────────
print('=' * 55)
print('  Training LSTM Autoencoder (Anomaly Detection)')
print('=' * 55)

anomaly_agent = DLAnomalyDetectionAgent(
    input_path=f'{OUTPUT_DIR}/model_ready_data.csv',
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    anomaly_percentile=90,
    dropout_rate=0.2,
    output_dir=OUTPUT_DIR,
    show_plots=False
)

anomaly_results, detected_anomalies, anomaly_summary = anomaly_agent.run_train(
    epochs=ANOMALY_EPOCHS, lr=LR
)

print('\n✅ Anomaly model trained!')
print(f'   Total records   : {anomaly_summary["total_records_analyzed"]}')
print(f'   Anomalies found : {anomaly_summary["total_anomalies_detected"]}')
print(f'   Anomaly rate    : {anomaly_summary["anomaly_rate"]*100:.2f}%')
print(f'   Risk breakdown  : {anomaly_summary["risk_distribution"]}')

## 💡 Section 6 – Recommendations, Validation & Report

In [ ]:
# ── Run downstream agents for FORECASTING ─────────────────────────────────────
print('▶ Forecasting pipeline – downstream agents...')

rec_f = RecommendationAgent(
    mode='forecasting', future_df=future_df, peaks=peaks,
    forecasting_summary=forecasting_summary
)
rec_output_f = rec_f.run(output_path=f'{OUTPUT_DIR}/recommendation_report_forecasting.json')

human_f = HumanInLoopAgent(recommendation_output=rec_output_f,
                           output_path=f'{OUTPUT_DIR}/human_validation_forecasting.json')
human_output_f = human_f.run()

report_f = ReportGenerationAgent(
    mode='forecasting', forecasting_summary=forecasting_summary,
    recommendation_output=rec_output_f, human_validation_output=human_output_f,
    output_dir=OUTPUT_DIR
)
final_report_f = report_f.run()

print('✅ Forecasting report generated!')
print(f'   Validation status: {human_output_f["validation_summary"]["final_status"]}')

In [ ]:
# ── Run downstream agents for ANOMALY ─────────────────────────────────────────
print('▶ Anomaly pipeline – downstream agents...')

rec_a = RecommendationAgent(
    mode='anomaly', detected_anomalies=detected_anomalies,
    anomaly_summary=anomaly_summary
)
rec_output_a = rec_a.run(output_path=f'{OUTPUT_DIR}/recommendation_report_anomaly.json')

human_a = HumanInLoopAgent(recommendation_output=rec_output_a,
                           output_path=f'{OUTPUT_DIR}/human_validation_anomaly.json')
human_output_a = human_a.run()

report_a = ReportGenerationAgent(
    mode='anomaly', anomaly_summary=anomaly_summary,
    recommendation_output=rec_output_a, human_validation_output=human_output_a,
    output_dir=OUTPUT_DIR
)
final_report_a = report_a.run()

print('✅ Anomaly report generated!')
print(f'   Validation status: {human_output_a["validation_summary"]["final_status"]}')

## 📊 Section 7 – Visualizations

In [ ]:
# ── Display all saved plots ────────────────────────────────────────────────────
plots = [
    (f'{OUTPUT_DIR}/forecasting_training_loss.png', '📉 Forecasting LSTM – Training vs Validation Loss'),
    (f'{OUTPUT_DIR}/forecasting_future_peaks.png',  '📈 Future Demand Forecast & Peak Detection'),
    (f'{OUTPUT_DIR}/anomaly_training_loss.png',     '📉 Anomaly Autoencoder – Training vs Validation Loss'),
    (f'{OUTPUT_DIR}/anomaly_detected_points.png',   '🔴 Anomaly Detection Results'),
]

for path, title in plots:
    if os.path.exists(path):
        print(f'\n{title}')
        display(IPImage(path))
    else:
        print(f'⚠️  {title} – file not found')

In [ ]:
# ── Interactive matplotlib charts ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Historical sales (from model-ready)
model_df_plot = pd.read_csv(f'{OUTPUT_DIR}/model_ready_data.csv', parse_dates=['date'])
ax = axes[0, 0]
ax.plot(model_df_plot['date'], model_df_plot['sales'], linewidth=0.8)
ax.set_title('Historical Daily Sales'); ax.set_xlabel('Date'); ax.set_ylabel('Sales')
ax.tick_params(axis='x', rotation=30)

# 2. Future demand forecast
ax = axes[0, 1]
ax.plot(future_df['date'], future_df['predicted_sales'], marker='o', ms=4, label='Forecast')
ax.axhline(y=future_df['peak_threshold'].iloc[0], ls='--', color='orange', label='Peak threshold')
if len(peaks) > 0:
    ax.scatter(peaks['date'], peaks['predicted_sales'], color='red', s=70, zorder=5, label='Peak')
ax.set_title('Future Demand Forecast'); ax.set_xlabel('Date'); ax.set_ylabel('Sales')
ax.legend(); ax.tick_params(axis='x', rotation=30)

# 3. Anomaly reconstruction error
ax = axes[1, 0]
anomaly_results_plot = pd.read_csv(f'{OUTPUT_DIR}/anomaly_detection_results.csv', parse_dates=['date'])
ax.plot(anomaly_results_plot['date'], anomaly_results_plot['reconstruction_error'], linewidth=0.8, label='Error')
ax.axhline(y=anomaly_results_plot['anomaly_threshold'].iloc[0], ls='--', color='red', label='Threshold')
ax.set_title('Reconstruction Error (Anomaly Model)'); ax.set_xlabel('Date'); ax.set_ylabel('MSE Error')
ax.legend(); ax.tick_params(axis='x', rotation=30)

# 4. Anomaly type distribution
ax = axes[1, 1]
type_dist = anomaly_summary['anomaly_type_distribution']
if type_dist:
    ax.bar(list(type_dist.keys()), list(type_dist.values()), color='steelblue', edgecolor='white')
    ax.set_title('Anomaly Type Distribution'); ax.set_xlabel('Type'); ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
else:
    ax.text(0.5, 0.5, 'No anomalies detected', ha='center', va='center')
    ax.set_title('Anomaly Type Distribution')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/summary_dashboard.png', dpi=120)
plt.show()
print('📊 Dashboard saved to outputs/summary_dashboard.png')

## 📋 Section 8 – View Final Report

In [ ]:
# Print the text business report for the forecasting pipeline
report_path = f'{OUTPUT_DIR}/final_business_report.txt'
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())

In [ ]:
# ── Quick metrics summary table ────────────────────────────────────────────────
print('\n📊 FORECASTING MODEL METRICS')
print('-' * 35)
for k, v in forecasting_summary['metrics'].items():
    if isinstance(v, (int, float)):
        print(f'  {k:<25} {v:.4f}')

print('\n📊 ANOMALY DETECTION SUMMARY')
print('-' * 35)
print(f'  Total records analyzed:  {anomaly_summary["total_records_analyzed"]}')
print(f'  Anomalies detected:      {anomaly_summary["total_anomalies_detected"]}')
print(f'  Anomaly rate:            {anomaly_summary["anomaly_rate"]*100:.2f}%')
print(f'  Risk distribution:       {anomaly_summary["risk_distribution"]}')
print(f'  Anomaly types:           {anomaly_summary["anomaly_type_distribution"]}')

## 🔄 Section 9 – Inference on New Data (Optional)

If you want to run predictions on new data **without retraining**, use this section.

In [ ]:
# ── INFERENCE MODE ─────────────────────────────────────────────────────────────
# Set RUN_INFERENCE = True to run inference on new data
RUN_INFERENCE = False   # ← change to True if needed
INFERENCE_MODE = 'forecasting'   # 'forecasting' or 'anomaly'

if RUN_INFERENCE:
    print(f'🔄 Running INFERENCE mode: {INFERENCE_MODE}')

    # Upload new data
    print('Upload new data CSV(s):')
    new_uploaded = files.upload()
    os.makedirs('data/new_data', exist_ok=True)
    for fname, data in new_uploaded.items():
        with open(f'data/new_data/{fname}', 'wb') as f:
            f.write(data)

    # Preprocess
    new_prep = RetailPreprocessingAgent(input_path='data/new_data')
    _, new_model_df = new_prep.run(
        clean_output_path=f'{OUTPUT_DIR}/new_clean_data.csv',
        model_output_path=f'{OUTPUT_DIR}/new_model_ready_data.csv'
    )

    if INFERENCE_MODE == 'forecasting':
        inf_agent = ForecastingTrendAgent(
            input_path=f'{OUTPUT_DIR}/new_model_ready_data.csv',
            window_size=WINDOW_SIZE, future_steps=FUTURE_STEPS,
            output_dir=OUTPUT_DIR
        )
        future_df_inf, peaks_inf, summary_inf = inf_agent.run_inference()
        print('✅ Forecasting inference done!')
        display(future_df_inf.head())
    else:
        inf_agent = DLAnomalyDetectionAgent(
            input_path=f'{OUTPUT_DIR}/new_model_ready_data.csv',
            window_size=WINDOW_SIZE, output_dir=OUTPUT_DIR
        )
        results_inf, anomalies_inf, summary_inf = inf_agent.run_inference()
        print('✅ Anomaly inference done!')
        display(anomalies_inf.head())
else:
    print('ℹ️  Inference mode is OFF. Set RUN_INFERENCE = True to use.')

## 💾 Section 10 – Download All Outputs

In [ ]:
import zipfile

zip_path = 'retail_pipeline_outputs.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zipf.write(fpath, fname)

print(f'📦 Zip created: {zip_path}')
print('Contents:')
with zipfile.ZipFile(zip_path) as zf:
    for name in sorted(zf.namelist()):
        print(f'  {name}')

files.download(zip_path)
print('✅ Download started!')